# Trabajo Práctico Nro. 1B 
### Ing. Javier Ouret - UCA - Facultad de Ingeniería
## Ejemplos de Raw Sockets usando bash kernel para su compilación.

### ¿Qué es un Raw Socket?

Es un tipo especial de socket que permite enviar y recibir paquetes sin procesar a nivel de red, saltándose las abstracciones del sistema operativo.   
Se pueden crear paquetes IP, TCP, UDP o ICMP personalizados.   
Para qué se usan:
- Construcción de herramientas de red (ping, traceroute, nmap).
- Implementación de protocolos personalizados.
- Monitoreo y análisis de tráfico.
- Pruebas de seguridad (ethical hacking).

| Característica         | Detalle                                                  |
| ---------------------- | -------------------------------------------------------- |
| Nivel de operación     | Capa de red (OSI capa 3) o superior (capa 4).            |
| Permisos necesarios    | Root (por razones de seguridad).                         |
| Encabezado | IP, TCP, UDP, ICMP. |
| OS          | Linux y BSD (limitado en Windows).                       |


#### Creación de Raw Sockets en C

Para poder ver protocolo IP sin procesar ETH_P_IP = 0x0800, uprotocolo para paquetes IP (IPv4).

**IMPORTANTE: para usar raw sockets se necesitan privilegios de superusuario (root), ya que puede
generar tráfico malicioso.**
Raw sockets permiten falsificación de IP, sniffing, y otros ataques. 
- Algunos sistemas limitan su uso.
- Usarlos sólo en entornos controlados y con fines educativos o de prueba.

#### Control del Encabezado IP

#### Estructura de un Paquete IP Personalizado

### A) Ping Básico. Usa ICMP Echo Request.

#### Estructura del paquete ICMP (Echo Request)

#### Estructura del header IP:

#### Estructura del header ICMP:

#### Cálculo del checksum en Python (igual que en C)

#### Código en C para enviar un paquete ICMP

In [1]:
from pathlib import Path
import textwrap

code = textwrap.dedent(r'''#include <stdio.h>
#include <string.h>
#include <stdlib.h>
#include <sys/socket.h>
#include <arpa/inet.h>
#include <netinet/ip.h>
#include <netinet/ip_icmp.h>
#include <unistd.h>
#include <errno.h>
#include <sys/time.h>

#define PACKET_SIZE 4096
#define DEST_IP "127.0.0.1"
#define MAX_PINGS 5

unsigned short checksum(void *b, int len) {
    unsigned short *buf = b;
    unsigned int sum = 0;
    for (; len > 1; len -= 2)
        sum += *buf++;
    if (len == 1)
        sum += *(unsigned char*)buf;
    sum = (sum >> 16) + (sum & 0xFFFF);
    sum += (sum >> 16);
    return ~sum;
}

int main() {
    char packet[PACKET_SIZE];
    struct sockaddr_in dest;
    int sockfd, one = 1;
    struct timeval timeout = {1, 0};

    sockfd = socket(AF_INET, SOCK_RAW, IPPROTO_ICMP);
    if (sockfd < 0) {
        perror("socket");
        return 1;
    }

    setsockopt(sockfd, IPPROTO_IP, IP_HDRINCL, &one, sizeof(one));
    setsockopt(sockfd, SOL_SOCKET, SO_RCVTIMEO, &timeout, sizeof(timeout));

    dest.sin_family = AF_INET;
    dest.sin_addr.s_addr = inet_addr(DEST_IP);

    for (int seq = 1; seq <= MAX_PINGS; seq++) {
        memset(packet, 0, PACKET_SIZE);

        struct iphdr *ip = (struct iphdr *) packet;
        struct icmphdr *icmp = (struct icmphdr *) (packet + sizeof(struct iphdr));

        ip->ihl = 5;
        ip->version = 4;
        ip->tos = 0;
        ip->tot_len = htons(sizeof(struct iphdr) + sizeof(struct icmphdr));
        ip->id = htons(1234 + seq);
        ip->frag_off = 0;
        ip->ttl = 64;
        ip->protocol = IPPROTO_ICMP;
        ip->saddr = inet_addr("127.0.0.1");
        ip->daddr = dest.sin_addr.s_addr;
        ip->check = checksum((unsigned short *)ip, sizeof(struct iphdr));

        icmp->type = ICMP_ECHO;
        icmp->code = 0;
        icmp->un.echo.id = htons(1234);
        icmp->un.echo.sequence = htons(seq);
        icmp->checksum = checksum((unsigned short *)icmp, sizeof(struct icmphdr));

        if (sendto(sockfd, packet, ntohs(ip->tot_len), 0,
                   (struct sockaddr *)&dest, sizeof(dest)) < 0) {
            perror("sendto");
        } else {
            printf("Ping #%d enviado a %s...\n", seq, DEST_IP);
        }

        char recv_buffer[PACKET_SIZE];
        struct sockaddr_in recv_addr;
        socklen_t addr_len = sizeof(recv_addr);

        ssize_t bytes = recvfrom(sockfd, recv_buffer, sizeof(recv_buffer), 0,
                                 (struct sockaddr *)&recv_addr, &addr_len);

        if (bytes < 0) {
            if (errno == EAGAIN || errno == EWOULDBLOCK)
                printf("Timeout esperando respuesta ICMP #%d\n", seq);
            else
                perror("recvfrom");
        } else {
            struct iphdr *recv_ip = (struct iphdr *)recv_buffer;
            struct icmphdr *recv_icmp = (struct icmphdr *)(recv_buffer + (recv_ip->ihl * 4));

            if (recv_icmp->type == ICMP_ECHOREPLY) {
                printf("Respuesta ICMP #%d recibida desde %s\n\n",
                       ntohs(recv_icmp->un.echo.sequence),
                       inet_ntoa(recv_addr.sin_addr));
            } else {
                printf("Recibido otro tipo de paquete ICMP: tipo %d\n", recv_icmp->type);
            }
        }

        sleep(1);
    }

    close(sockfd);
    return 0;
}
''')

Path("raw_sockets_ping.c").write_text(code)
print("Archivo generado: raw_sockets_ping.c")


Archivo generado: raw_sockets_ping.c


**NOTA: Cerrar las ventanas de las Terminales al finalizar su uso**

Compilo el código usando Bash C:

In [2]:
import subprocess
subprocess.run(["gcc", "--version"], check=False)


gcc (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



CompletedProcess(args=['gcc', '--version'], returncode=0)

In [3]:
import subprocess
subprocess.run(["gcc", "raw_sockets_ping.c", "-o", "raw_sockets_ping"], check=False)


CompletedProcess(args=['gcc', 'raw_sockets_ping.c', '-o', 'raw_sockets_ping'], returncode=0)

Verifico que el ejecutable esté disponible:

In [4]:
from pathlib import Path
for p in sorted(Path('.').iterdir()):
    print(p.name)


COMPARACION_SNIFFERS.md
Cliente_Servidor_No_Bloqueante_C.ipynb
Cliente_Servidor_Websockets_R1.ipynb
Códigos en C
Códigos en Python
PROGRESO_TP1B.md
README_TP1C.md
Revisiones Anteriores
TAREA5_Estructuras_Datos.md
TP1A_Cliente_Servidor_R1.ipynb
TP1A_Cliente_Servidor_R1_backup.ipynb
TP1A_Cliente_Servidor_R2.ipynb
TP1B_Raw_Sockets_R2.ipynb
TP1C_Sockets_Python.ipynb
__pycache__
archivo_prueba.txt
archivo_recibido_1775518443.txt
archivo_recibido_1775518576.txt
archivo_recibido_select_1775518539.txt
archivo_recibido_select_1775518585.txt
cliente_bloqueante
cliente_bloqueante.c
cliente_concurrente_archivos.py
cliente_icmp_personalizado
cliente_icmp_personalizado.c
cliente_select
cliente_select.c
cliente_select_archivos.py
cliente_tcp_bash
cliente_tcp_bash.c
cliente_tcp_c.ipynb
demo_sniffer_continuo.sh
demo_tarea3_cliente_sniffer.sh
demo_tarea4_logging.sh
demo_tarea6_multiprotocolo.sh
demo_tarea7_interfaz_web.sh
icmp_packet.bin
interfaz_web_sniffer.py
prueba_tp1c.py
raw_sockets_ping
raw_socket

La terminal de Jupyter puede funcionar mal. Para ejecutarlo correctamente abro ventana por afuera del Jupyter Notebook:

In [5]:
print("Ejecutar manualmente en terminal externa:")
print("sudo ./raw_sockets_ping")


Ejecutar manualmente en terminal externa:
sudo ./raw_sockets_ping


Capturarlo con tcpdump en la interfaz lo

In [6]:
print("Ejecutar manualmente en otra terminal:")
print("sudo tcpdump -i lo -nv icmp")


Ejecutar manualmente en otra terminal:
sudo tcpdump -i lo -nv icmp


#### Visualizar el paquete en Python

In [7]:
import binascii

# Supongamos que el paquete ya fue capturado en formato binario
packet = b'\x45\x00\x00\x1c\x30\x39\x00\x00\x40\x01\xa6\xec\xc0\xa8\x00\x64\x08\x08\x08\x08\x08\x00\xf7\xff\x04\xd2\x00\x01'

# Mostrar byte a byte
print("Paquete (hex):")
for i in range(0, len(packet), 2):
    print(binascii.hexlify(packet[i:i+2]), end=' ')


Paquete (hex):
b'4500' b'001c' b'3039' b'0000' b'4001' b'a6ec' b'c0a8' b'0064' b'0808' b'0808' b'0800' b'f7ff' b'04d2' b'0001' 

### B) Sniffer ICMP.

Este programa crea un socket RAW para recibir todos los paquetes IP y filtra los ICMP. 
- Se guarda el contenido del paquete en un archivo binario para luego analizarlo con Python.

In [8]:
from pathlib import Path
import textwrap

code = textwrap.dedent(r'''#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <sys/socket.h>
#include <arpa/inet.h>
#include <netinet/ip.h>
#include <netinet/ip_icmp.h>

int main() {
    int sockfd;
    char buffer[65536];
    struct sockaddr saddr;
    socklen_t saddr_len = sizeof(saddr);

    sockfd = socket(AF_INET, SOCK_RAW, IPPROTO_ICMP);
    if (sockfd < 0) {
        perror("Socket error");
        return 1;
    }

    printf("Sniffer ICMP iniciado... (Ctrl+C para detener)\n");

    while (1) {
        ssize_t packet_size = recvfrom(sockfd, buffer, sizeof(buffer), 0, &saddr, &saddr_len);
        if (packet_size < 0) {
            perror("recvfrom error");
            continue;
        }

        struct iphdr *ip = (struct iphdr*) buffer;
        if (ip->protocol == IPPROTO_ICMP) {
            struct icmphdr *icmp = (struct icmphdr*)(buffer + ip->ihl * 4);
            printf("ICMP capturado: Tipo=%d Codigo=%d ID=%d Secuencia=%d\n",
                icmp->type,
                icmp->code,
                ntohs(icmp->un.echo.id),
                ntohs(icmp->un.echo.sequence)
            );

            FILE *f = fopen("icmp_packet.bin", "wb");
            if (f) {
                fwrite(buffer, 1, packet_size, f);
                fclose(f);
                printf("Paquete guardado en icmp_packet.bin\n");
                break;
            }
        }
    }

    close(sockfd);
    return 0;
}
''')

Path("raw_sockets_sniffer.c").write_text(code)
print("Archivo generado: raw_sockets_sniffer.c")


Archivo generado: raw_sockets_sniffer.c


Compilo el código usando Bash C:

In [9]:
import subprocess
subprocess.run(["gcc", "raw_sockets_sniffer.c", "-o", "raw_sockets_sniffer"], check=False)


CompletedProcess(args=['gcc', 'raw_sockets_sniffer.c', '-o', 'raw_sockets_sniffer'], returncode=0)

In [10]:
print("Ejecutar manualmente en terminal externa:")
print("sudo ./raw_sockets_sniffer")


Ejecutar manualmente en terminal externa:
sudo ./raw_sockets_sniffer


#### Ejecutar ping 127.0.0.1 o la IP local o remota desde otra terminal para generar tráfico ICMP.

#### Leer y visualizar el paquete en Python

In [11]:
import struct
import binascii

with open("icmp_packet.bin", "rb") as f:
    data = f.read()

print("Paquete capturado:")
print(binascii.hexlify(data, sep=' '))

# Extraer y mostrar cabeceras IP e ICMP
ip_header = data[:20]
ip_fields = struct.unpack("!BBHHHBBH4s4s", ip_header)

print("\n IP Header:")
print(f"Version: {ip_fields[0] >> 4}")
print(f"IHL: {ip_fields[0] & 0xF} ({(ip_fields[0] & 0xF)*4} bytes)")
print(f"Total Length: {ip_fields[2]}")
print(f"Protocol: {ip_fields[6]} (should be 1 for ICMP)")
print(f"Source IP: {'.'.join(map(str, ip_fields[8]))}")
print(f"Destination IP: {'.'.join(map(str, ip_fields[9]))}")

icmp_offset = (ip_fields[0] & 0xF) * 4
icmp_header = data[icmp_offset:icmp_offset+8]
icmp_fields = struct.unpack("!BBHHH", icmp_header)

print("\nICMP Header:")
print(f"Type: {icmp_fields[0]}")
print(f"Code: {icmp_fields[1]}")
print(f"Checksum: {hex(icmp_fields[2])}")
print(f"ID: {icmp_fields[3]}")
print(f"Sequence: {icmp_fields[4]}")

Paquete capturado:
b'45 00 00 20 a4 9e 00 00 ff 01 19 3c 7f 00 00 01 7f 00 00 01 08 00 6b 95 d2 04 01 00 48 6f 6c 61'

 IP Header:
Version: 4
IHL: 5 (20 bytes)
Total Length: 32
Protocol: 1 (should be 1 for ICMP)
Source IP: 127.0.0.1
Destination IP: 127.0.0.1

ICMP Header:
Type: 8
Code: 0
Checksum: 0x6b95
ID: 53764
Sequence: 256


Ejemplo de captura:

Visualización gráfica del paquete binario con matplotlib.   
Se muestra:
- Los bytes como valores hexadecimales.
- Un mapa de calor (heatmap) donde cada byte es un píxel codificado por color.


**Instalar matplotlib**

In [12]:
import importlib.util

if importlib.util.find_spec("matplotlib") is None or importlib.util.find_spec("numpy") is None:
    print("matplotlib/numpy no estan instalados; se omite visualizacion grafica.")
else:
    import matplotlib.pyplot as plt
    import numpy as np

    with open("icmp_packet.bin", "rb") as f:
        packet = f.read()

    bytes_list = list(packet)
    length = len(bytes_list)
    cols = 16
    rows = (length + cols - 1) // cols

    padded = bytes_list + [0] * (rows * cols - length)
    matrix = np.array(padded).reshape((rows, cols))

    plt.figure(figsize=(12, 5))
    plt.imshow(matrix, cmap='viridis', aspect='auto')
    plt.title("Mapa de bytes del paquete ICMP")
    plt.colorbar(label="Valor del byte")
    plt.xlabel("Byte en columna")
    plt.ylabel("Fila del paquete")

    for i in range(rows):
        for j in range(cols):
            byte_val = matrix[i, j]
            plt.text(j, i, f'{byte_val:02X}', ha='center', va='center', color='white', fontsize=8)

    plt.tight_layout()
    plt.show()


matplotlib/numpy no estan instalados; se omite visualizacion grafica.


### Enviar paquete TCP SYN con Raw Socket en C

In [13]:
from pathlib import Path
import textwrap

code = textwrap.dedent(r'''#include <stdio.h>
#include <string.h>
#include <stdlib.h>
#include <unistd.h>
#include <arpa/inet.h>
#include <sys/socket.h>
#include <netinet/ip.h>
#include <netinet/tcp.h>

unsigned short checksum(unsigned short *ptr, int nbytes) {
    long sum = 0;
    unsigned short oddbyte;
    unsigned short answer;

    while (nbytes > 1) {
        sum += *ptr++;
        nbytes -= 2;
    }

    if (nbytes == 1) {
        oddbyte = 0;
        *((unsigned char*)&oddbyte) = *(unsigned char*)ptr;
        sum += oddbyte;
    }

    sum = (sum >> 16) + (sum & 0xffff);
    sum += (sum >> 16);
    answer = ~sum;
    return answer;
}

struct pseudo_header {
    unsigned int source_address;
    unsigned int dest_address;
    unsigned char placeholder;
    unsigned char protocol;
    unsigned short tcp_length;
};

int main() {
    int sock = socket(AF_INET, SOCK_RAW, IPPROTO_TCP);
    if (sock < 0) {
        perror("socket error");
        return 1;
    }

    char datagram[4096], *data, *pseudogram;
    memset(datagram, 0, 4096);

    struct iphdr *iph = (struct iphdr *)datagram;
    struct tcphdr *tcph = (struct tcphdr *)(datagram + sizeof(struct iphdr));

    struct sockaddr_in sin;
    sin.sin_family = AF_INET;
    sin.sin_port = htons(80);
    sin.sin_addr.s_addr = inet_addr("127.0.0.1");

    data = datagram + sizeof(struct iphdr) + sizeof(struct tcphdr);
    strcpy(data, "");

    iph->ihl = 5;
    iph->version = 4;
    iph->tos = 0;
    iph->tot_len = sizeof(struct iphdr) + sizeof(struct tcphdr);
    iph->id = htons(54321);
    iph->frag_off = 0;
    iph->ttl = 64;
    iph->protocol = IPPROTO_TCP;
    iph->saddr = inet_addr("127.0.0.1");
    iph->daddr = sin.sin_addr.s_addr;
    iph->check = checksum((unsigned short *)datagram, iph->tot_len);

    tcph->source = htons(1234);
    tcph->dest = htons(80);
    tcph->seq = htonl(0);
    tcph->ack_seq = 0;
    tcph->doff = 5;
    tcph->syn = 1;
    tcph->window = htons(5840);
    tcph->check = 0;
    tcph->urg_ptr = 0;

    struct pseudo_header psh;
    psh.source_address = iph->saddr;
    psh.dest_address = iph->daddr;
    psh.placeholder = 0;
    psh.protocol = IPPROTO_TCP;
    psh.tcp_length = htons(sizeof(struct tcphdr));

    int psize = sizeof(struct pseudo_header) + sizeof(struct tcphdr);
    pseudogram = malloc(psize);
    if (!pseudogram) {
        perror("malloc failed");
        close(sock);
        return 1;
    }

    memcpy(pseudogram, &psh, sizeof(struct pseudo_header));
    memcpy(pseudogram + sizeof(struct pseudo_header), tcph, sizeof(struct tcphdr));

    tcph->check = checksum((unsigned short *)pseudogram, psize);

    int one = 1;
    if (setsockopt(sock, IPPROTO_IP, IP_HDRINCL, &one, sizeof(one)) < 0) {
        perror("setsockopt error");
        free(pseudogram);
        close(sock);
        return 1;
    }

    if (sendto(sock, datagram, iph->tot_len, 0,
               (struct sockaddr *)&sin, sizeof(sin)) < 0) {
        perror("sendto failed");
    } else {
        printf("Paquete TCP SYN enviado a %s\n", inet_ntoa(sin.sin_addr));
    }

    free(pseudogram);
    close(sock);
    return 0;
}
''')

Path("raw_sockets_tcp.c").write_text(code)
print("Archivo generado: raw_sockets_tcp.c")


Archivo generado: raw_sockets_tcp.c


### Consigna del TP 1B
**Ejercicios a realizar durante la clase. Incluir los códigos dentro de este mismo Notebook**

Con esta información es posible crear un simple "sniffer", que muestre todo el contenido de los paquetes TCP que se reciban. ( En este ejemplo se evitan los headers IP y TCP, y se imprime solamente el "payload" con encabezados IP y TCP contenidos en el paquete ).
Usar también tcpdump que es una herramienta de línea de comandos en sistemas Unix/Linux que sirve para capturar y analizar tráfico de red. Funciona a nivel de capa de enlace (nivel 2 del modelo OSI) y utiliza libpcap para interceptar paquetes.

- Utilizando el código "sniffer" que es un programa que muestra el contenido del tráfico que llega, modificarlo para explicar cada paso que va ejecutando en pantalla.
- Modificar el sniffer para que vaya mostrando todo el tráfico que llega.
- Enviar tráfico al "sniffer" desde el cliente escrito en la parte A del TP1
- Enviar tráfico ICMP al "sniffer" y mostrar los resultados con un LOG con comentarios.
- Explicar las estructuras del código de ICMP y del Sniffer, especialmente los campos de protocolo.
- Estudiar el "sniffer" para que muestre IP, TCP y UDP.
- Mostrar resultados.


## TAREA 1: Modificar sniffer ICMP con explicaciones paso a paso

Utilizando el código "sniffer" original, lo modificamos para explicar cada paso que va ejecutando en pantalla.

A continuación implementamos el código modificado:

## TAREA 1: Modificar sniffer ICMP con explicaciones paso a paso

Utilizando el código "sniffer" original, lo modificamos para explicar cada paso que va ejecutando en pantalla.

In [14]:
from pathlib import Path
src = Path("raw_sockets_sniffer_explicado.c")
if src.exists():
    print("Usando archivo existente:", src)
    print("Tamanio:", src.stat().st_size, "bytes")
else:
    print("No existe raw_sockets_sniffer_explicado.c. Ejecutar primero la celda de generacion correspondiente.")


Usando archivo existente: raw_sockets_sniffer_explicado.c
Tamanio: 4329 bytes


Compilar el sniffer modificado con explicaciones:

In [15]:
import subprocess
subprocess.run(["gcc", "raw_sockets_sniffer_explicado.c", "-o", "raw_sockets_sniffer_explicado"], check=False)


CompletedProcess(args=['gcc', 'raw_sockets_sniffer_explicado.c', '-o', 'raw_sockets_sniffer_explicado'], returncode=0)

Ejecutar el sniffer modificado (abre nueva terminal):

In [16]:
print("Ejecutar manualmente en terminal externa:")
print("sudo ./raw_sockets_sniffer_explicado")


Ejecutar manualmente en terminal externa:
sudo ./raw_sockets_sniffer_explicado


## TAREA 1: Modificar sniffer ICMP con explicaciones paso a paso

Utilizando el código "sniffer" original, lo modificamos para explicar cada paso que va ejecutando en pantalla.

In [17]:
from pathlib import Path
src = Path("raw_sockets_sniffer_explicado.c")
if src.exists():
    print("Usando archivo existente:", src)
    print("Tamanio:", src.stat().st_size, "bytes")
else:
    print("No existe raw_sockets_sniffer_explicado.c. Ejecutar primero la celda de generacion correspondiente.")


Usando archivo existente: raw_sockets_sniffer_explicado.c
Tamanio: 4329 bytes


Compilar el sniffer modificado con explicaciones:

In [18]:
import subprocess
subprocess.run(["gcc", "raw_sockets_sniffer_explicado.c", "-o", "raw_sockets_sniffer_explicado"], check=False)


CompletedProcess(args=['gcc', 'raw_sockets_sniffer_explicado.c', '-o', 'raw_sockets_sniffer_explicado'], returncode=0)

Ejecutar el sniffer modificado (abre nueva terminal):

In [19]:
print("Ejecutar manualmente en terminal externa:")
print("sudo ./raw_sockets_sniffer_explicado")


Ejecutar manualmente en terminal externa:
sudo ./raw_sockets_sniffer_explicado


#### Para raw_sockets_tcp.c
- El programa anterior arma paquete IP + TCP.
- Lo envía como un SYN (inicio de conexión TCP).
- Calcula los checksums (IP y TCP).
- Requiere IP de origen válida.
- Probar con IPs distintas de localhost
- Verlo con tcpdump
- Mientras se ejecuta el programa, en otra terminal ejecutar tcpdump (reemplazar por interfaz correcta si es necesario)

sudo tcpdump -i lo or -i wlp1s0 tcp

## Verificacion automatica TP1B (kernel Python)

Estas celdas ejecutan una validacion reproducible en este entorno:

1. Compilan los sniffers del TP1B.
2. Ejecutan pruebas con privilegios (`sudo -n`) y trafico real (ICMP/TCP/UDP).
3. Verifican salida, log y paquete binario capturado.

> Nota: se agregan como bloque nuevo para no modificar el material original del notebook.

In [20]:
import subprocess
from pathlib import Path

BASE = Path('/workspaces/PDI_TP1-TP2-TP3-TP4/TP1')


def sh(cmd: str, check: bool = True):
    print(f"$ {cmd}")
    p = subprocess.run(
        cmd,
        shell=True,
        cwd=BASE,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Comando fallo con exit code {p.returncode}: {cmd}")
    return p

print('Directorio de trabajo:', BASE)
print('Compilando sniffers...')
sh('gcc raw_sockets_sniffer_explicado.c -o /tmp/sniff_exp')
sh('gcc raw_sockets_sniffer_logging.c -o /tmp/sniff_log')
sh('gcc raw_sockets_sniffer_multiprotocolo.c -o /tmp/sniff_multi')
print('OK: compilacion finalizada')

Directorio de trabajo: /workspaces/PDI_TP1-TP2-TP3-TP4/TP1
Compilando sniffers...
$ gcc raw_sockets_sniffer_explicado.c -o /tmp/sniff_exp
$ gcc raw_sockets_sniffer_logging.c -o /tmp/sniff_log


$ gcc raw_sockets_sniffer_multiprotocolo.c -o /tmp/sniff_multi
OK: compilacion finalizada


In [21]:
# Prueba funcional automatizada con trafico real
# Requiere sudo -n habilitado para raw sockets.

script = r'''
set -e
cd /workspaces/PDI_TP1-TP2-TP3-TP4/TP1

sudo -n true

# 1) Sniffer explicado + cliente ICMP
sudo -n timeout 10s stdbuf -oL -eL /tmp/sniff_exp > /tmp/out_exp_nb.txt 2>&1 & pid1=$!
sudo -n ./cliente_icmp_personalizado 127.0.0.1 > /tmp/out_icmp_nb1.txt 2>&1 || true
wait $pid1 || true

# 2) Sniffer logging + cliente ICMP
sudo -n timeout 10s stdbuf -oL -eL /tmp/sniff_log > /tmp/out_log_nb.txt 2>&1 & pid2=$!
sudo -n ./cliente_icmp_personalizado 127.0.0.1 > /tmp/out_icmp_nb2.txt 2>&1 || true
wait $pid2 || true

# 3) Sniffer multiprotocolo + trafico ICMP/TCP/UDP
sudo -n timeout 12s stdbuf -oL -eL /tmp/sniff_multi > /tmp/out_multi_nb.txt 2>&1 & pid3=$!
sudo -n ./cliente_icmp_personalizado 127.0.0.1 > /tmp/out_icmp_nb3.txt 2>&1 || true
python3 - << 'PY'
import socket
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(1)
try:
    s.connect(("127.0.0.1", 80))
except Exception:
    pass
s.close()

u = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
u.sendto(b"hola-udp", ("127.0.0.1", 9999))
u.close()
print("trafico TCP/UDP enviado")
PY
wait $pid3 || true

printf "=== EXP ===\n"
head -n 40 /tmp/out_exp_nb.txt || true
printf "\n=== LOG ===\n"
head -n 20 /tmp/out_log_nb.txt || true
printf "\n=== MULTI ===\n"
head -n 80 /tmp/out_multi_nb.txt || true
printf "\n=== LOG FILE (tail) ===\n"
tail -n 30 trafico_icmp_log.txt || true
'''

sh(script)

$ 
set -e
cd /workspaces/PDI_TP1-TP2-TP3-TP4/TP1

sudo -n true

# 1) Sniffer explicado + cliente ICMP
sudo -n timeout 10s stdbuf -oL -eL /tmp/sniff_exp > /tmp/out_exp_nb.txt 2>&1 & pid1=$!
sudo -n ./cliente_icmp_personalizado 127.0.0.1 > /tmp/out_icmp_nb1.txt 2>&1 || true
wait $pid1 || true

# 2) Sniffer logging + cliente ICMP
sudo -n timeout 10s stdbuf -oL -eL /tmp/sniff_log > /tmp/out_log_nb.txt 2>&1 & pid2=$!
sudo -n ./cliente_icmp_personalizado 127.0.0.1 > /tmp/out_icmp_nb2.txt 2>&1 || true
wait $pid2 || true

# 3) Sniffer multiprotocolo + trafico ICMP/TCP/UDP
sudo -n timeout 12s stdbuf -oL -eL /tmp/sniff_multi > /tmp/out_multi_nb.txt 2>&1 & pid3=$!
sudo -n ./cliente_icmp_personalizado 127.0.0.1 > /tmp/out_icmp_nb3.txt 2>&1 || true
python3 - << 'PY'
import socket
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(1)
try:
    s.connect(("127.0.0.1", 80))
except Exception:
    pass
s.close()

u = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
u.sendto(b"hola-udp", (

trafico TCP/UDP enviado
=== EXP ===
=== PASO 1: Creación del socket RAW ===
Creando socket RAW para capturar paquetes ICMP...
✓ Socket RAW creado exitosamente (sockfd = 3)
  - Familia: AF_INET (IPv4)
  - Tipo: SOCK_RAW (acceso directo a paquetes)
  - Protocolo: IPPROTO_ICMP (solo paquetes ICMP)

=== PASO 2: Inicio del bucle de captura ===
Sniffer ICMP iniciado... Esperando paquetes (Ctrl+C para detener)

=== PASO 3: Esperando recepción de paquetes ===
Llamando a recvfrom() para recibir el siguiente paquete...

=== LOG ===
=== SNIFFER ICMP - MODO LOGGING DETALLADO ===
Guardando TODO el tráfico ICMP en 'trafico_icmp_log.txt'
Presione Ctrl+C para detener

=== PAQUETE ICMP #1 [2026-04-14 00:06:18] ===
📦 Tamaño: 32 bytes | Origen: 127.0.0.1 → Destino: 127.0.0.1
🔍 ICMP Tipo: 8 | Código: 0
✓ Log detallado guardado en archivo


=== MULTI ===
=== SNIFFER MULTI-PROTOCOLO ===
Capturando paquetes ICMP, TCP y UDP simultáneamente
Presione Ctrl+C para detener

✓ Socket creado para ICMP (protocolo 1)


CompletedProcess(args='\nset -e\ncd /workspaces/PDI_TP1-TP2-TP3-TP4/TP1\n\nsudo -n true\n\n# 1) Sniffer explicado + cliente ICMP\nsudo -n timeout 10s stdbuf -oL -eL /tmp/sniff_exp > /tmp/out_exp_nb.txt 2>&1 & pid1=$!\nsudo -n ./cliente_icmp_personalizado 127.0.0.1 > /tmp/out_icmp_nb1.txt 2>&1 || true\nwait $pid1 || true\n\n# 2) Sniffer logging + cliente ICMP\nsudo -n timeout 10s stdbuf -oL -eL /tmp/sniff_log > /tmp/out_log_nb.txt 2>&1 & pid2=$!\nsudo -n ./cliente_icmp_personalizado 127.0.0.1 > /tmp/out_icmp_nb2.txt 2>&1 || true\nwait $pid2 || true\n\n# 3) Sniffer multiprotocolo + trafico ICMP/TCP/UDP\nsudo -n timeout 12s stdbuf -oL -eL /tmp/sniff_multi > /tmp/out_multi_nb.txt 2>&1 & pid3=$!\nsudo -n ./cliente_icmp_personalizado 127.0.0.1 > /tmp/out_icmp_nb3.txt 2>&1 || true\npython3 - << \'PY\'\nimport socket\ns = socket.socket(socket.AF_INET, socket.SOCK_STREAM)\ns.settimeout(1)\ntry:\n    s.connect(("127.0.0.1", 80))\nexcept Exception:\n    pass\ns.close()\n\nu = socket.socket(socket

In [22]:
# Validacion de artefactos generados
import binascii
import struct
from pathlib import Path

bin_path = BASE / 'icmp_packet.bin'
log_path = BASE / 'trafico_icmp_log.txt'

print('icmp_packet.bin existe:', bin_path.exists())
print('trafico_icmp_log.txt existe:', log_path.exists())

if bin_path.exists():
    data = bin_path.read_bytes()
    print('Tamanio paquete:', len(data), 'bytes')
    print('Hex paquete:', binascii.hexlify(data, sep=b' ').decode())

    if len(data) >= 28:
        ip_fields = struct.unpack('!BBHHHBBH4s4s', data[:20])
        ihl = (ip_fields[0] & 0x0F) * 4
        print('IP version:', ip_fields[0] >> 4)
        print('IHL:', ihl)
        print('IP protocolo:', ip_fields[6])
        icmp = struct.unpack('!BBHHH', data[ihl:ihl+8])
        print('ICMP tipo/codigo/id/seq:', icmp[0], icmp[1], icmp[3], icmp[4])

if log_path.exists():
    lines = log_path.read_text(errors='ignore').splitlines()
    print('\nUltimas 12 lineas de log:')
    for line in lines[-12:]:
        print(line)

icmp_packet.bin existe: True
trafico_icmp_log.txt existe: True
Tamanio paquete: 32 bytes
Hex paquete: 45 00 00 20 a4 9e 00 00 ff 01 19 3c 7f 00 00 01 7f 00 00 01 08 00 6b 95 d2 04 01 00 48 6f 6c 61
IP version: 4
IHL: 20
IP protocolo: 1
ICMP tipo/codigo/id/seq: 8 0 53764 256

Ultimas 12 lineas de log:
CHECKSUM IP: 0xaa7b
ID PAQUETE IP: 4959
TIPO ICMP: 8 (Echo Request - Ping enviado)
CÓDIGO ICMP: 0
CHECKSUM ICMP: 0x6b95
ID ECHO: 53764
SECUENCIA ECHO: 256
TAMAÑO PAYLOAD ICMP: 4 bytes
PAYLOAD (hex): 48 6f 6c 61 
PAYLOAD (texto): Hola
----------------------------------------

